# IT135IU · Course Project · D2: Data and Profiling

**Topic 15 (own topic): Prioritizing Cardiovascular Screening Outreach, CDC BRFSS 2023**

| Member | Student ID | Role |
|---|---|---|
| Vũ Tiến Dũng | ITDSIU25011 | Product Owner |
| Mai Hoàng Gia Bảo | ITDSIU25005 | Data Engineer |
| Lê Nguyễn Trúc Phương | ITDSIU25029 | Data Analyst |
| Nguyễn Trần Phước Lộc | ITDSIU25020 | Visualisation Lead |
| Huỳnh Bá Dũng | ITDSIU25012 | Reproducibility Lead |
| Dương Minh Chính | ITDSIU25007 | Ethics and Privacy Officer |

**Decision question.** Which adult population groups should a public-health program prioritize for cardiovascular screening
and prevention outreach, based on combinations of major cardiovascular risk factors and reported cardiovascular disease burden?

**This notebook delivers (D2):** (1) the dataset loaded and merged, (2) a data profile table, (3) a cleaning log,
(4) five data manipulation operations, each with its output. Statistical analysis and charts belong to D3.

*Data source: Centers for Disease Control and Prevention (CDC), Behavioral Risk Factor Surveillance System (BRFSS) 2023.
This analysis was produced by students of International University, VNU-HCM and has not been reviewed or endorsed by CDC, HHS or the U.S. Government.*

## 0 · Setup
Run the notebook from top to bottom (**Restart kernel → Run All**). Inputs: the working dataset built by `notebooks/01_data_preparation.ipynb`
(`data/processed/brfss2023_working.parquet`, raw CDC codes) and the small lookup table `data/lookup/state_lookup.csv`.

In [1]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import hashlib, json

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42                       # fixed seed; D2 uses no randomness, kept for D3
np.random.seed(SEED)
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WORKING = ROOT / "data/processed/brfss2023_working.parquet"
LOOKUP  = ROOT / "data/lookup/state_lookup.csv"
DOCS    = ROOT / "docs"
OUTDIR  = DOCS / "d2_outputs"
OUTDIR.mkdir(parents=True, exist_ok=True)
RETRIEVAL_DATE = "20 Oct 2026"  # keep identical to data/README.md, section 2

def ver(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"

print("pandas", pd.__version__, "| numpy", np.__version__, "| pyarrow", ver("pyarrow"), "| jinja2", ver("jinja2"), "| seed", SEED)
print("project root:", ROOT)

pandas 3.0.6 | numpy 2.5.4 | pyarrow 26.0.0 | jinja2 3.1.6 | seed 42
project root: /Users/tiendung/Downloads/Intro_ds-main


## 1 · Load the dataset and record its source

In [2]:
assert WORKING.exists(), f"Missing {WORKING}. Get the file from the team drive or run notebooks/01_data_preparation.ipynb (see data/README.md)."
df_raw = pd.read_parquet(WORKING)

manifest = json.loads((DOCS / "working_dataset_manifest.json").read_text(encoding="utf-8"))
verify   = json.loads((DOCS / "source_verification.json").read_text(encoding="utf-8"))
sha = hashlib.sha256(WORKING.read_bytes()).hexdigest()

assert df_raw.shape == (manifest["rows"], manifest["columns"]), df_raw.shape
print(f"Loaded working dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print("File hash matches manifest:", sha == manifest["output_sha256"], "(a rebuilt file with another pyarrow version may differ; values were re-checked by M2)")

source = pd.DataFrame({"item": [
    "Dataset", "Publisher", "Landing page", "Original file", "Original file size (bytes)", "Original file SHA-256",
    "Rows in public file", "Columns in public file", "Columns kept for this project", "Retrieval date", "Terms of use"],
  "value": [
    "BRFSS 2023, Combined Landline and Cell Phone data (LLCP2023)",
    "U.S. Centers for Disease Control and Prevention (CDC)",
    "https://www.cdc.gov/brfss/annual_data/annual_2023.html",
    "LLCP2023XPT.zip (SAS Transport), read from LLCP2023.XPT",
    f"{verify['files']['xpt']['bytes']:,} (XPT)",
    verify["files"]["xpt"]["sha256"],
    f"{verify['checks']['rows']['observed']:,}",
    f"{verify['n_columns']['observed']}",
    f"{manifest['columns']}",
    RETRIEVAL_DATE,
    "CDC public-domain material: attribute CDC/BRFSS and state that use implies no CDC/HHS endorsement (https://www.cdc.gov/other/agencymaterials.html)"]})
display(source.style.hide(axis="index"))

Loaded working dataset: 433,323 rows x 29 columns
File hash matches manifest: True (a rebuilt file with another pyarrow version may differ; values were re-checked by M2)


item,value
Dataset,"BRFSS 2023, Combined Landline and Cell Phone data (LLCP2023)"
Publisher,U.S. Centers for Disease Control and Prevention (CDC)
Landing page,https://www.cdc.gov/brfss/annual_data/annual_2023.html
Original file,"LLCP2023XPT.zip (SAS Transport), read from LLCP2023.XPT"
Original file size (bytes),"1,205,554,400 (XPT)"
Original file SHA-256,3d3bf8ef5195bde227828ddc4c90745b76e8b304f8f5b9a043b6d99895fd1615
Rows in public file,"433,323"
Columns in public file,350
Columns kept for this project,29
Retrieval date,20 Oct 2026


### 1.1 · Independent re-check of the file against CDC's published figures
The data engineer already verified the working file against the CDC codebook (`docs/source_verification.json`). The analyst repeats the check **from the file itself**,
so that every later number rests on a verified base: (a) frequencies of the four outcome variables, (b) the sum of the weights, and
(c) weighted percentages that CDC prints in its codebook ("Weighted %" column, reproduced in `docs/variable_dictionary.md`).
A weighted percentage that matches CDC to two decimals shows that the codes, the weights and the rows are intact.

In [3]:
# (a) frequencies of the outcome variables as printed in the CDC 2023 codebook ("NA" = blank)
CODEBOOK_COUNTS = {
    "CVDINFR4": {1: 23451, 2: 407304, 7: 2314, 9: 251, "NA": 3},
    "CVDCRHD4": {1: 23454, 2: 405638, 7: 3936, 9: 292, "NA": 3},
    "CVDSTRK3": {1: 18350, 2: 413499, 7: 1212, 9: 258, "NA": 4},
    "_MICHD":   {1: 36311, 2: 392427, "NA": 4585},
}
rows = []
for v, expected in CODEBOOK_COUNTS.items():
    for k, n_exp in expected.items():
        n_obs = int(df_raw[v].isna().sum()) if k == "NA" else int((df_raw[v] == k).sum())
        rows.append({"variable": v, "code": k, "codebook": n_exp, "file": n_obs, "match": n_exp == n_obs})
cnt = pd.DataFrame(rows)
print(f"(a) Outcome frequencies: {int(cnt['match'].sum())} of {len(cnt)} codes match the CDC codebook exactly")

# (b) weights
w = df_raw["_LLCPWT"].to_numpy()
recorded = verify["checks"]["weight"]["sum_of_weights"]
n_eff = w.sum() ** 2 / (w ** 2).sum()
print(f"(b) Sum of weights: {w.sum():,.0f} | recorded by the data engineer: {recorded:,.0f} | match: {bool(np.isclose(w.sum(), recorded))}")
print(f"    Effective sample size from unequal weights (Kish): {n_eff:,.0f} of {len(w):,} respondents, design effect {len(w) / n_eff:.2f} (before clustering)")

# (c) weighted percentages vs the codebook. CDC's denominator is every respondent with a non-blank answer.
CODEBOOK_WEIGHTED = [   # (variable, code, weighted % printed by CDC, meaning)
    ("CVDINFR4", 1, 4.16, "Ever told: heart attack"),
    ("CVDCRHD4", 1, 4.03, "Ever told: angina or coronary heart disease"),
    ("CVDSTRK3", 1, 3.37, "Ever told: stroke"),
    ("_MICHD", 1, 6.47, "Heart attack or coronary heart disease"),
    ("_RFHYPE6", 2, 34.28, "Told they have high blood pressure"),
    ("_RFCHOL3", 2, 37.20, "Told cholesterol is high"),
    ("DIABETE4", 1, 12.04, "Diagnosed diabetes"),
    ("_BMI5CAT", 4, 32.80, "Obese (BMI 30 or more)"),
    ("_RFSMOK3", 2, 10.68, "Current smoker"),
    ("_TOTINDA", 2, 24.71, "No leisure-time physical activity"),
    ("_AGEG5YR", 1, 12.37, "Aged 18-24"),
    ("_AGEG5YR", 13, 5.24, "Aged 80 or older"),
]
rows = []
for v, code_, pct_cdc, what in CODEBOOK_WEIGHTED:
    answered = df_raw[v].notna().to_numpy()
    hit = (df_raw[v] == code_).fillna(False).to_numpy(bool)
    pct_file = 100 * w[hit & answered].sum() / w[answered].sum()
    rows.append({"variable": v, "code": code_, "meaning": what, "cdc_weighted_pct": pct_cdc,
                 "file_weighted_pct": pct_file, "difference": pct_file - pct_cdc, "match": abs(pct_file - pct_cdc) <= 0.011})
wt = pd.DataFrame(rows)
print(f"(c) Weighted percentages: {int(wt['match'].sum())} of {len(wt)} reproduce the CDC codebook to two decimals")
display(wt.style.format({"cdc_weighted_pct": "{:.2f}", "file_weighted_pct": "{:.2f}", "difference": "{:+.3f}"}).hide(axis="index"))
wt.to_csv(OUTDIR / "d2_integrity_checks.csv", index=False)

(a) Outcome frequencies: 18 of 18 codes match the CDC codebook exactly
(b) Sum of weights: 254,139,829 | recorded by the data engineer: 254,139,829 | match: True
    Effective sample size from unequal weights (Kish): 84,602 of 433,323 respondents, design effect 5.12 (before clustering)
(c) Weighted percentages: 12 of 12 reproduce the CDC codebook to two decimals


variable,code,meaning,cdc_weighted_pct,file_weighted_pct,difference,match
CVDINFR4,1,Ever told: heart attack,4.16,4.16,+0.004,True
CVDCRHD4,1,Ever told: angina or coronary heart disease,4.03,4.03,+0.002,True
CVDSTRK3,1,Ever told: stroke,3.37,3.37,+0.005,True
_MICHD,1,Heart attack or coronary heart disease,6.47,6.47,+0.004,True
_RFHYPE6,2,Told they have high blood pressure,34.28,34.28,+0.001,True
_RFCHOL3,2,Told cholesterol is high,37.20,37.20,+0.005,True
DIABETE4,1,Diagnosed diabetes,12.04,12.04,-0.001,True
_BMI5CAT,4,Obese (BMI 30 or more),32.80,32.80,-0.001,True
_RFSMOK3,2,Current smoker,10.68,10.68,-0.003,True
_TOTINDA,2,No leisure-time physical activity,24.71,24.71,-0.004,True


## 2 · Data profile
Profile of the **raw working dataset** (CDC codes, nothing recoded yet). Each variable is listed with its plain-language `label` from the CDC codebook
and its project `role`. `special_code_rows` counts rows holding an "unknown" code for that variable (7 = Don't know, 9 = Refused, 14 or 9 for some calculated variables;
blanks are counted in `n_missing`); `pct_unusable` = blanks + unknown codes, as a share of all rows.

In [4]:
dictionary = pd.read_csv(DOCS / "variable_dictionary.csv", encoding="utf-8-sig")
role = dictionary.set_index("variable")["role"]
label = dictionary.set_index("variable")["label"]

# Codes that mean "unknown / refused" for each variable (from the CDC codebook, see docs/variable_dictionary.md)
UNKNOWN_CODES = {
    "CVDINFR4": [7, 9], "CVDCRHD4": [7, 9], "CVDSTRK3": [7, 9], "BPHIGH6": [7, 9], "_RFHYPE6": [9],
    "CHOLCHK3": [7, 9], "TOLDHI3": [7, 9], "_RFCHOL3": [9], "DIABETE4": [7, 9], "_RFBMI5": [9],
    "_SMOKER3": [9], "_RFSMOK3": [9], "EXERANY2": [7, 9], "_TOTINDA": [9], "_AGEG5YR": [14],
    "_INCOMG1": [9], "_EDUCAG": [9], "_RACE": [9],
}

def profile(d):
    rows = []
    for c in d.columns:
        s = d[c]
        numeric = pd.api.types.is_numeric_dtype(s)
        rows.append({
            "variable": c, "label": label.get(c, ""), "role": role.get(c, ""), "dtype": str(s.dtype), "n_rows": len(s),
            "n_missing": int(s.isna().sum()), "pct_missing": 100 * s.isna().mean(),
            "special_code_rows": int(s.isin(UNKNOWN_CODES.get(c, [])).sum()),
            "n_unique": int(s.nunique()),
            "min": s.min() if numeric else np.nan, "max": s.max() if numeric else np.nan})
    return pd.DataFrame(rows)

prof = profile(df_raw)
prof["pct_unusable"] = prof["pct_missing"] + 100 * prof["special_code_rows"] / prof["n_rows"]
print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]} | Total weight (_LLCPWT): {df_raw['_LLCPWT'].sum():,.0f}")
display(prof.drop(columns="n_rows").style.format({"pct_missing": "{:.2f}", "pct_unusable": "{:.2f}", "min": "{:,.1f}", "max": "{:,.1f}"})
        .set_properties(subset=["label"], **{"text-align": "left"}).hide(axis="index"))
prof.to_csv(OUTDIR / "d2_data_profile.csv", index=False)

Rows: 433,323 | Columns: 29 | Total weight (_LLCPWT): 254,139,829


variable,label,role,dtype,n_missing,pct_missing,special_code_rows,n_unique,min,max,pct_unusable
CVDINFR4,Ever Diagnosed with Heart Attack,outcome,Int64,3,0.00,2565,4,1.0,9.0,0.59
CVDCRHD4,Ever Diagnosed with Angina or Coronary Heart Disease,outcome,Int64,3,0.00,4228,4,1.0,9.0,0.98
CVDSTRK3,Ever Diagnosed with a Stroke,outcome,Int64,4,0.00,1470,4,1.0,9.0,0.34
_MICHD,Ever had CHD or MI,outcome,Int64,4585,1.06,0,2,1.0,2.0,1.06
BPHIGH6,Ever Told Blood Pressure High,risk_factor,Int64,3,0.00,1916,6,1.0,9.0,0.44
_RFHYPE6,High Blood Pressure Calculated Variable,risk_factor,Int64,0,0.00,1919,3,1.0,9.0,0.44
CHOLCHK3,How Long since Cholesterol Checked,risk_factor,Int64,3,0.00,26446,9,1.0,9.0,6.10
TOLDHI3,Ever Told Cholesterol Is High,risk_factor,Int64,51812,11.96,3272,4,1.0,9.0,12.71
_RFCHOL3,High Cholesterol Calculated Variable,risk_factor,Int64,51811,11.96,3273,3,1.0,9.0,12.71
DIABETE4,(Ever told) you had diabetes,risk_factor,Int64,5,0.00,979,6,1.0,9.0,0.23


In [5]:
by_role = prof.groupby("role").agg(variables=("variable", "count"), mean_pct_unusable=("pct_unusable", "mean")).round(2)
display(by_role)
worst = prof.sort_values("pct_unusable", ascending=False).head(6)[["variable", "role", "pct_unusable"]]
print("Variables with the most unusable values (these drive the later loss of rows):")
display(worst.style.format({"pct_unusable": "{:.2f}"}).hide(axis="index"))

,variables,mean_pct_unusable
role,,
demographic,9,2.73
outcome,4,0.74
risk_factor,13,5.53
survey_design,3,0.00


Variables with the most unusable values (these drive the later loss of rows):


variable,role,pct_unusable
_INCOMG1,demographic,19.99
TOLDHI3,risk_factor,12.71
_RFCHOL3,risk_factor,12.71
_BMI5,risk_factor,9.35
_BMI5CAT,risk_factor,9.35
_RFBMI5,risk_factor,9.35


### 2.1 · The three continuous variables
Everything else in the file is a category stored as a code. Only age, BMI and the weight are real quantities, so they get a numeric summary.

In [6]:
cont = pd.DataFrame({"_AGE80 (years, 80 = 80 or older)": df_raw["_AGE80"].astype(float),
                     "BMI (_BMI5 / 100)": df_raw["_BMI5"].astype(float) / 100,
                     "_LLCPWT (final weight)": df_raw["_LLCPWT"].astype(float)})
display(cont.describe(percentiles=[.25, .5, .75]).T.style.format("{:,.2f}"))

,count,mean,std,min,25%,50%,75%,max
"_AGE80 (years, 80 = 80 or older)","433,323.00",55.38,17.91,18.00,41.00,58.00,71.00,80.00
BMI (_BMI5 / 100),"392,788.00",28.48,6.54,12.02,24.10,27.40,31.62,99.84
_LLCPWT (final weight),"433,323.00",586.49,"1,190.72",0.16,103.37,258.27,604.55,"69,785.82"


### 2.2 · What the profile tells us
1. **Structure.** 433,323 respondents and 29 columns. 28 columns are integer CDC codes; only the weight `_LLCPWT` is a decimal. No column is empty or constant.
2. **The outcome is well measured.** Unusable answers for the three conditions are only 0.34% (stroke), 0.59% (heart attack) and 0.98% (angina or coronary heart disease).
   `_MICHD` has 1.06% blanks and leaves out stroke, so `CVD_ANY` is built from the three components instead.
3. **The risk factors carry most of the missing data.** Three of them will later drive the loss of rows: cholesterol 12.71% (never checked or unknown), BMI 9.35% and smoking 5.32%.
   Income (19.99%) is the worst variable overall but is not part of `RiskCount`.
4. **Codes are not numbers.** 7 (don't know), 9 (refused) and 14 (age unknown) sit inside numeric columns, so a mean or a comparison on raw codes would be wrong
   (`_AGEG5YR` runs from 1 to 14, and 14 does not mean "older than 80+"). Section 5 recodes every such code and logs it.
5. **Cholesterol is missing by design.** A blank `_RFCHOL3` means the person never had cholesterol checked, or did not know or refused to say. It does not mean normal cholesterol,
   so these respondents stay unknown instead of "No" (cleaning log C5).
6. **Weights are very unequal.** `_LLCPWT` runs from about 0.2 to about 69,800 and sums to 254,139,829, so one respondent stands for very different numbers of adults.
   Only weighted figures describe the population (section 6). `_STSTR` and `_PSU` are present for the uncertainty work in D3.
7. **Units.** `_BMI5` stores BMI × 100: 1,202 to 9,984 means BMI 12.0 to 99.8.

## 3 · Operation 1 · MERGE with the state lookup table
`data/lookup/state_lookup.csv` (54 rows) is joined to the 433k respondents on the state FIPS code, giving each respondent a state name and a Census region.
The join must be many-to-one and must not change the number of rows.

In [7]:
lookup = pd.read_csv(LOOKUP)
assert lookup["state_fips"].is_unique

n_before = len(df_raw)
df = df_raw.merge(lookup, how="left", left_on="_STATE", right_on="state_fips",
                  validate="many_to_one", indicator=True)
assert len(df) == n_before, "merge changed the number of rows"
assert (df["_merge"] == "both").all(), "some _STATE codes have no match in the lookup"
df = df.drop(columns=["_merge", "state_fips"])

print(f"Rows before merge: {n_before:,} | after merge: {len(df):,} | unmatched: 0")
print(f"Geographies in the data: {df['_STATE'].nunique()} of {len(lookup)} lookup rows")
display(df.groupby("census_region").agg(respondents=("_STATE", "size"), states=("state_usps", "nunique")).sort_values("respondents", ascending=False))
display(df[["_STATE", "state_usps", "state_name", "census_region", "_AGEG5YR", "CVDINFR4", "_LLCPWT"]].head())

absent = sorted(set(lookup["state_name"]) - set(df["state_name"]))
print("Lookup rows with no respondents:", ", ".join(absent))
assert absent == ["Kentucky", "Pennsylvania"], absent      # CDC did not release these two states in the 2023 public file
print("= the two states CDC did not release in the 2023 public file: the data hold 48 states + DC + 3 territories.")

Rows before merge: 433,323 | after merge: 433,323 | unmatched: 0
Geographies in the data: 52 of 54 lookup rows


,respondents,states
census_region,,
Midwest,119119,12
West,114376,13
South,113273,16
Northeast,78338,8
Territory,8217,3


,_STATE,state_usps,state_name,census_region,_AGEG5YR,CVDINFR4,_LLCPWT
0,1,AL,Alabama,South,13,2,605.427887
1,1,AL,Alabama,South,13,2,1121.992705
2,1,AL,Alabama,South,13,2,600.963308
3,1,AL,Alabama,South,12,2,605.427887
4,1,AL,Alabama,South,12,2,281.711042


Lookup rows with no respondents: Kentucky, Pennsylvania
= the two states CDC did not release in the 2023 public file: the data hold 48 states + DC + 3 territories.


## 4 · Operation 2 · FILTER to the population of interest, with the first cleaning-log entries
The decision question concerns **U.S. adults**. BRFSS also surveys three territories (Guam, Puerto Rico, U.S. Virgin Islands), which are excluded
so that results can be described as "U.S. adults in the 48 states and DC present in the 2023 public file" (Kentucky and Pennsylvania were not released by CDC).

The **cleaning log** is built step by step with `log_*` helpers; each entry records the variable, the original code, the new value, the reason and the rows affected.

In [8]:
cleaning_log = []

def log(step, variable, original, new, reason, n_rows):
    cleaning_log.append({"step": step, "variable": variable, "original": original, "new_value": new,
                         "reason": reason, "rows_affected": int(n_rows),
                         "pct_of_rows": 100 * int(n_rows) / len(df_raw)})

# C1: territories
terr = df["census_region"].eq("Territory")
log("C1", "_STATE", "66 Guam, 72 Puerto Rico, 78 U.S. Virgin Islands", "row removed",
    "Outside the 48 states + DC in the file; results are described as U.S. adults", terr.sum())
n_all = len(df)
df = df.loc[~terr].reset_index(drop=True)
print(f"Rows: {n_all:,} -> {len(df):,}  (removed {n_all - len(df):,} territory respondents)")

# C2: exact duplicate rows (there is no respondent ID in the public file, so all 29 raw columns are compared)
dups = df_raw.duplicated().sum()
log("C2", "all 29 columns", "identical rows", "none removed",
    "Checked for duplicates; weight and PSU make true duplicates implausible", dups)
print("Exact duplicate rows:", dups)

# C3: plausibility ranges
checks = {
    "_AGE80 in 18..80": df["_AGE80"].between(18, 80).all(),
    "_LLCPWT > 0": (df["_LLCPWT"] > 0).all(),
    "_BMI5 (x100) in 1200..9999 when present": df["_BMI5"].dropna().between(1200, 9999).all(),
}
for k, v in checks.items():
    print(f"{k}: {'ok' if v else 'FAILED'}")
    assert v, k
log("C3", "_AGE80, _LLCPWT, _BMI5", "range check", "no change", "All values inside the plausible range; no row dropped", 0)

Rows: 433,323 -> 425,106  (removed 8,217 territory respondents)
Exact duplicate rows: 0
_AGE80 in 18..80: ok
_LLCPWT > 0: ok
_BMI5 (x100) in 1200..9999 when present: ok


## 5 · Operation 3 · DERIVED COLUMNS: risk flags, `CVD_ANY`, `RiskCount`, age bands
Rule agreed for the whole project: **unknown is never "No"**. `Yes -> 1`, `No -> 0`, everything else (don't know, refused, blank) `-> <NA>`.
`recode()` applies a mapping and logs every code it touches, with the number of rows.

In [9]:
def recode(src, mapping, new, step, reason_by_code, default_reason="Unknown or missing: kept as NA, not coded as No"):
    # Map CDC codes of `src` to `new` (nullable Int8). Codes not in `mapping` (including blanks) become <NA>. Every code is logged.
    s = df[src]
    out = s.map(mapping)                       # unmapped codes and <NA> become NaN
    df[new] = out.astype("Int8")
    for code_, n in s.value_counts(dropna=False).sort_index().items():
        label = "blank" if pd.isna(code_) else int(code_)
        new_val = mapping.get(code_, pd.NA) if not pd.isna(code_) else pd.NA
        new_txt = "<NA>" if pd.isna(new_val) else int(new_val)
        reason = reason_by_code.get(label, default_reason)
        log(step, f"{src} -> {new}", f"{label}", new_txt, reason, n)

# --- six risk-factor flags (1 = risk factor present)
recode("_RFHYPE6", {1: 0, 2: 1}, "HTN_FLAG", "C4",
       {1: "No high blood pressure. CDC code 1 also holds 'pregnancy only' and 'borderline' answers (BPHIGH6 = 2 or 4); kept as 0 per CDC definition",
        2: "Told they have high blood pressure"})
recode("_RFCHOL3", {1: 0, 2: 1}, "CHOL_FLAG", "C5",
       {1: "Cholesterol not reported high", 2: "Told cholesterol is high",
        "blank": "Never had cholesterol checked or unknown: NOT the same as normal cholesterol, kept as NA"})
recode("DIABETE4", {1: 1, 2: 0, 3: 0, 4: 0}, "DIABETES_FLAG", "C6",
       {1: "Diagnosed diabetes", 2: "Diabetes only during pregnancy: not chronic diabetes, coded 0",
        3: "No diabetes", 4: "Pre-diabetes / borderline: not diabetes, coded 0"})
recode("_BMI5CAT", {1: 0, 2: 0, 3: 0, 4: 1}, "OBESITY_FLAG", "C7",
       {1: "Underweight: not obese", 2: "Normal weight: not obese", 3: "Overweight but not obese (BMI < 30)", 4: "Obese (BMI >= 30)",
        "blank": "BMI not computable (height or weight missing): kept as NA"})
recode("_RFSMOK3", {1: 0, 2: 1}, "SMOKING_FLAG", "C8",
       {1: "Not a current smoker (former and never smokers)", 2: "Current smoker (every day or some days)"})
recode("_TOTINDA", {1: 0, 2: 1}, "INACTIVE_FLAG", "C9",
       {1: "Had leisure-time physical activity in the past 30 days", 2: "No leisure-time physical activity in the past 30 days"})

FLAGS = ["HTN_FLAG", "CHOL_FLAG", "DIABETES_FLAG", "OBESITY_FLAG", "SMOKING_FLAG", "INACTIVE_FLAG"]
print("Flag prevalence among respondents with a known value (unweighted):")
display(pd.DataFrame({"known": df[FLAGS].notna().sum(), "pct_present": 100 * df[FLAGS].mean()}).round(2))

Flag prevalence among respondents with a known value (unweighted):


,known,pct_present
HTN_FLAG,423197,40.77
CHOL_FLAG,370717,41.97
DIABETES_FLAG,424140,13.73
OBESITY_FLAG,385028,33.03
SMOKING_FLAG,402297,11.06
INACTIVE_FLAG,423862,24.4


In [10]:
# --- three outcome components: 1 = ever diagnosed, 0 = never, otherwise NA
OUT = {"CVDINFR4": "MI_FLAG", "CVDCRHD4": "CHD_FLAG", "CVDSTRK3": "STROKE_FLAG"}
for i, (src, new) in enumerate(OUT.items()):
    recode(src, {1: 1, 2: 0}, new, f"C1{0+i}",
           {1: "Reported ever diagnosed", 2: "Reported never diagnosed"})

# --- CVD_ANY: any of MI / CHD (angina or coronary heart disease) / stroke
comp = df[list(OUT.values())]
any_yes = comp.eq(1).fillna(False).any(axis=1).astype(bool)          # <NA> comparisons count as "not Yes"
all_no  = comp.eq(0).fillna(False).all(axis=1).astype(bool)          # all three explicitly "No" (an <NA> is NOT a No)
df["CVD_ANY"] = pd.Series(pd.NA, index=df.index, dtype="Int8")
df.loc[any_yes, "CVD_ANY"] = 1
df.loc[~any_yes & all_no, "CVD_ANY"] = 0
log("C13", "CVD_ANY", "MI, CHD or stroke = Yes", 1, "Reported at least one of the three conditions (a 'Yes' counts even if another answer is missing)", any_yes.sum())
log("C13", "CVD_ANY", "all three = No", 0, "Reported none of the three", (~any_yes & all_no).sum())
log("C13", "CVD_ANY", "no 'Yes' and at least one unknown", "<NA>", "Cannot tell: excluded from prevalence, not counted as No", df["CVD_ANY"].isna().sum())
print(df["CVD_ANY"].value_counts(dropna=False).sort_index().rename({0: "0 (none)", 1: "1 (any CVD)"}).to_frame("respondents"))

# --- RiskCount: complete-case over the six flags (0..6), plus how many flags are unknown
df["RiskMissingCount"] = df[FLAGS].isna().sum(axis=1).astype("Int8")
df["RiskCount"] = df[FLAGS].astype("float64").sum(axis=1, min_count=len(FLAGS)).astype("Int8")      # NA unless all 6 flags are known
log("C14", "RiskCount", "any of the 6 flags unknown", "<NA>",
    "Complete-case rule: RiskCount only when all six flags are known; RiskMissingCount keeps the number of unknown flags", df["RiskCount"].isna().sum())

# --- age: 14 = don't know / refused / missing -> NA; five-year codes 1..13 -> labels and broader bands
AGE_LABELS = {1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39", 5: "40-44", 6: "45-49", 7: "50-54",
              8: "55-59", 9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"}
BAND = {**{k: "18-34" for k in (1, 2, 3)}, **{k: "35-44" for k in (4, 5)}, **{k: "45-54" for k in (6, 7)},
        **{k: "55-64" for k in (8, 9)}, **{k: "65+" for k in (10, 11, 12, 13)}}
df["AGE_5YR"]  = df["_AGEG5YR"].map(AGE_LABELS)
df["AGE_BAND"] = pd.Categorical(df["_AGEG5YR"].map(BAND), categories=["18-34", "35-44", "45-54", "55-64", "65+"], ordered=True)
for k, n in df["_AGEG5YR"].value_counts().sort_index().items():
    log("C15", "_AGEG5YR -> AGE_5YR / AGE_BAND", int(k), AGE_LABELS.get(k, "<NA>") + (f" / {BAND[k]}" if k in BAND else ""),
        "Age category label and five-band grouping" if k in AGE_LABELS else "Age unknown (code 14): kept as NA", n)

print("RiskCount distribution (unweighted, complete cases only):")
display(df["RiskCount"].value_counts(dropna=False).sort_index().to_frame("respondents"))

             respondents
CVD_ANY                 
0 (none)          372652
1 (any CVD)        47670
NaN                 4784
RiskCount distribution (unweighted, complete cases only):


,respondents
RiskCount,
0,71907
1,89702
2,79041
3,52223
4,25950
5,8422
6,838
<NA>,97023


### 5.1 · Analysis-ready sample (row flow)
How many respondents survive each requirement. Dropping rows is a **selection decision**, so every step is shown.

In [11]:
flow = [("Working dataset (raw CDC file, 29 columns kept)", len(df_raw)),
        ("In the 48 states + DC", len(df)),
        ("... and CVD_ANY known", int(df["CVD_ANY"].notna().sum())),
        ("... and all six risk flags known (RiskCount)", int((df["CVD_ANY"].notna() & df["RiskCount"].notna()).sum())),
        ("... and age known", int((df["CVD_ANY"].notna() & df["RiskCount"].notna() & df["_AGEG5YR"].between(1, 13)).sum()))]
flow = pd.DataFrame(flow, columns=["step", "respondents"])
flow["pct_of_working_dataset"] = 100 * flow["respondents"] / flow["respondents"].iloc[0]
flow["lost_vs_previous"] = (flow["respondents"].shift(1) - flow["respondents"]).fillna(0).astype(int)
display(flow.style.format({"respondents": "{:,}", "pct_of_working_dataset": "{:.1f}", "lost_vs_previous": "{:,}"}).hide(axis="index"))

df["IN_SAMPLE"] = (df["CVD_ANY"].notna() & df["RiskCount"].notna() & df["AGE_BAND"].notna())
log("C16", "IN_SAMPLE", "CVD_ANY, RiskCount and age all known", "kept", "Analysis-ready sample for D3 (complete case)", df["IN_SAMPLE"].sum())
log("C16", "IN_SAMPLE", "any of them unknown", "not in sample", "Complete-case limitation: respondents with a missing flag may differ from those without (see section 7)", (~df["IN_SAMPLE"]).sum())
sample = df.loc[df["IN_SAMPLE"]].copy()
print(f"Analysis-ready sample: {len(sample):,} respondents")

step,respondents,pct_of_working_dataset,lost_vs_previous
"Working dataset (raw CDC file, 29 columns kept)","433,323",100.0,0
In the 48 states + DC,"425,106",98.1,"8,217"
... and CVD_ANY known,"420,322",97.0,"4,784"
... and all six risk flags known (RiskCount),"325,109",75.0,"95,213"
... and age known,"321,978",74.3,"3,131"


Analysis-ready sample: 321,978 respondents


### 5.2 · Who is lost? A check on the complete-case rule
Most of the loss in 5.1 comes from respondents with an unknown risk flag. If they differ from the rest, the sample is not typical. Among respondents whose CVD status and age are known,
the table compares those kept with those dropped **only because a risk flag is unknown** (means and percentages use the survey weight).

In [12]:
known = df["CVD_ANY"].notna() & df["AGE_BAND"].notna()
sub = df.loc[known].copy()
sub["group"] = np.where(sub["IN_SAMPLE"], "In analysis sample", "Dropped (a risk flag is unknown)")

def group_stats(g):
    wg = g["_LLCPWT"].to_numpy(); y = g["CVD_ANY"].astype(float).to_numpy()
    return pd.Series({"respondents": len(g),
                      "mean_age_weighted": np.average(g["_AGE80"].astype(float), weights=wg),
                      "pct_aged_65plus_weighted": 100 * np.average((g["AGE_BAND"] == "65+").to_numpy(float), weights=wg),
                      "cvd_pct_unweighted": 100 * y.mean(),
                      "cvd_pct_weighted": 100 * np.average(y, weights=wg)})

stats = pd.DataFrame({name: group_stats(g) for name, g in sub.groupby("group")}).T
display(stats.style.format({"respondents": "{:,.0f}", "mean_age_weighted": "{:.1f}", "pct_aged_65plus_weighted": "{:.1f}",
                            "cvd_pct_unweighted": "{:.1f}", "cvd_pct_weighted": "{:.1f}"}))
stats.to_csv(OUTDIR / "d2_selection_check.csv")

a, b = stats.loc["In analysis sample"], stats.loc["Dropped (a risk flag is unknown)"]
print(f"Dropped respondents are on average {b['mean_age_weighted']:.1f} years old ({a['mean_age_weighted']:.1f} in the sample); "
      f"weighted reported-CVD prevalence is {b['cvd_pct_weighted']:.1f}% against {a['cvd_pct_weighted']:.1f}% in the sample.")

drop = sub.loc[~sub["IN_SAMPLE"]]
print(f"Which flags are unknown among the {len(drop):,} dropped respondents (a person can miss several):")
display(pd.DataFrame({"unknown": drop[FLAGS].isna().sum(), "pct_of_dropped": 100 * drop[FLAGS].isna().mean()}).round(1))

,respondents,mean_age_weighted,pct_aged_65plus_weighted,cvd_pct_unweighted,cvd_pct_weighted
Dropped (a risk flag is unknown),"90,987",40.4,14.2,8.0,5.7
In analysis sample,"321,978",51.0,26.9,12.4,9.9


Dropped respondents are on average 40.4 years old (51.0 in the sample); weighted reported-CVD prevalence is 5.7% against 9.9% in the sample.
Which flags are unknown among the 90,987 dropped respondents (a person can miss several):


,unknown,pct_of_dropped
HTN_FLAG,1456,1.6
CHOL_FLAG,52086,57.2
DIABETES_FLAG,633,0.7
OBESITY_FLAG,35846,39.4
SMOKING_FLAG,21124,23.2
INACTIVE_FLAG,1074,1.2


**Reading the table.** The respondents dropped by the complete-case rule are not a random slice. They are about ten years younger (mean age 40.4 against 51.0), only 14.2% are aged 65 or older
(26.9% in the sample), and their weighted reported-CVD prevalence is 5.7% against 9.9%. Age is very likely a large part of the gap (the dropped group has about half as many people aged 65 or older). Two consequences for D3:
(1) the **absolute** prevalence in the analysis sample is higher than in all U.S. adults, so the report should compare groups with each other (the age-by-risk table does this) rather than quote the sample level as a national figure;
(2) the comparison between risk groups should be shown **within age bands**, and a sensitivity run that allows one unknown flag (`RiskMissingCount` <= 1) should confirm that the pattern does not depend on the complete-case rule.
Unknown cholesterol (57% of those dropped), unknown BMI (39%) and unknown smoking status (23%) account for most of the loss.

### 5.3 · Profile of the derived columns (after cleaning)
The same profile idea applied to the new 0/1 columns, so that the cleaned data are described as carefully as the raw data. `pct_NA` is the share of unknown answers
(kept as `<NA>`, never as "No"); `pct_yes` is computed among respondents with a known answer, unweighted and weighted.

In [13]:
def derived_profile(d, cols):
    ww = d["_LLCPWT"].to_numpy(); rows = []
    for c in cols:
        y = d[c].astype("float64").to_numpy(); ok = ~np.isnan(y)
        rows.append({"variable": c, "n_known": int(ok.sum()), "n_NA": int((~ok).sum()), "pct_NA": 100 * (~ok).mean(),
                     "pct_yes_unweighted": 100 * y[ok].mean(), "pct_yes_weighted": 100 * np.sum(ww[ok] * y[ok]) / ww[ok].sum()})
    return pd.DataFrame(rows)

derived = derived_profile(df, FLAGS + ["MI_FLAG", "CHD_FLAG", "STROKE_FLAG", "CVD_ANY"])
display(derived.style.format({"n_known": "{:,}", "n_NA": "{:,}", "pct_NA": "{:.2f}", "pct_yes_unweighted": "{:.1f}", "pct_yes_weighted": "{:.1f}"}).hide(axis="index"))
derived.to_csv(OUTDIR / "d2_derived_profile.csv", index=False)

variable,n_known,n_NA,pct_NA,pct_yes_unweighted,pct_yes_weighted
HTN_FLAG,"423,197","1,909",0.45,40.8,34.4
CHOL_FLAG,"370,717","54,389",12.79,42.0,37.4
DIABETES_FLAG,"424,140",966,0.23,13.7,12.0
OBESITY_FLAG,"385,028","40,078",9.43,33.0,32.8
SMOKING_FLAG,"402,297","22,809",5.37,11.1,11.4
INACTIVE_FLAG,"423,862","1,244",0.29,24.4,24.5
MI_FLAG,"422,576","2,530",0.60,5.5,4.2
CHD_FLAG,"420,922","4,184",0.98,5.5,4.0
STROKE_FLAG,"423,651","1,455",0.34,4.3,3.4
CVD_ANY,"420,322","4,784",1.13,11.3,8.8


## 6 · Operations 4 and 5 · GROUP AND AGGREGATE
Descriptive tables only; the statistical relationship and charts are D3. `_LLCPWT` is the BRFSS final weight: a respondent stands for many adults, so
**weighted** figures describe the population and **raw n** only describes the survey. Weighted prevalence = Σ(weight × CVD_ANY) / Σ(weight).
These are weighted point estimates; strata (`_STSTR`) and PSU (`_PSU`) will be used for uncertainty in D3.
`pop_millions` is the weighted size of the analysis sample (respondents with complete data), **not** the whole U.S. adult population.

In [14]:
def summarise(d, by):
    g = d.assign(w=d["_LLCPWT"], wy=d["_LLCPWT"] * d["CVD_ANY"].astype(float), y=d["CVD_ANY"].astype(float)).groupby(by, observed=True)
    out = pd.DataFrame({
        "n_respondents": g.size(),
        "n_cvd": g["y"].sum().astype(int),
        "unweighted_prev_pct": 100 * g["y"].mean(),
        "weighted_prev_pct": 100 * g["wy"].sum() / g["w"].sum(),
        "pop_millions": g["w"].sum() / 1e6,
        "cvd_pop_millions": g["wy"].sum() / 1e6})
    return out

FMT = {"n_respondents": "{:,}", "n_cvd": "{:,}", "unweighted_prev_pct": "{:.1f}", "weighted_prev_pct": "{:.1f}",
       "pop_millions": "{:.1f}", "cvd_pop_millions": "{:.2f}"}

# Operation 4: by number of risk factors
by_risk = summarise(sample, "RiskCount")
print("Operation 4 · reported CVD by number of risk factors (RiskCount 0-6)")
display(by_risk.style.format(FMT))
by_risk.to_csv(OUTDIR / "d2_prevalence_by_riskcount.csv")

Operation 4 · reported CVD by number of risk factors (RiskCount 0-6)


,n_respondents,n_cvd,unweighted_prev_pct,weighted_prev_pct,pop_millions,cvd_pop_millions
RiskCount,,,,,,
0,"70,850","2,098",3.0,2.2,44.7,1.00
1,"88,218","6,311",7.2,5.4,51.0,2.75
2,"77,522","10,725",13.8,10.9,40.4,4.42
3,"51,063","10,501",20.6,18.6,25.2,4.71
4,"25,315","6,971",27.5,25.6,12.8,3.27
5,"8,193","2,841",34.7,33.3,4.2,1.39
6,817,318,38.9,39.7,0.4,0.16


In [15]:
# Operation 5: by age band x risk band (age is the main confounder, so the two are shown together)
sample = sample.assign(RISK_BAND=pd.Categorical(
    pd.cut(sample["RiskCount"].astype(int), bins=[-1, 1, 3, 6], labels=["0-1", "2-3", "4-6"]),
    categories=["0-1", "2-3", "4-6"], ordered=True))
by_age_risk = summarise(sample, ["AGE_BAND", "RISK_BAND"])
print("Operation 5 · weighted reported CVD prevalence (%) by age band and risk band")
display(by_age_risk["weighted_prev_pct"].unstack("RISK_BAND").style.format("{:.1f}"))
print("Respondents behind each cell (small cells are unstable)")
display(by_age_risk["n_respondents"].unstack("RISK_BAND").style.format("{:,}"))
by_age_risk.to_csv(OUTDIR / "d2_prevalence_by_age_and_risk.csv")

Operation 5 · weighted reported CVD prevalence (%) by age band and risk band


RISK_BAND,0-1,2-3,4-6
AGE_BAND,,,
18-34,0.9,2.8,10.9
35-44,2.0,4.7,12.9
45-54,2.4,8.9,22.5
55-64,4.7,14.5,28.6
65+,12.5,23.0,33.7


Respondents behind each cell (small cells are unstable)


RISK_BAND,0-1,2-3,4-6
AGE_BAND,,,
18-34,"31,039","7,831",585
35-44,"26,042","11,952","1,890"
45-54,"24,512","17,213","4,478"
55-64,"27,299","26,316","8,926"
65+,"50,176","65,273","18,446"


## 7 · Save the outputs and note what D2 shows and does not show

In [16]:
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(OUTDIR / "d2_cleaning_log.csv", index=False)
flow.to_csv(OUTDIR / "d2_sample_flow.csv", index=False)

CLEAN = ROOT / "data/processed/brfss2023_clean.parquet"      # git-ignored, shared with M4 / M6 through the team drive
df.to_parquet(CLEAN, index=False)
print(f"Cleaning log: {len(log_df)} entries -> {OUTDIR / 'd2_cleaning_log.csv'}")
print(f"Clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns -> {CLEAN.name}")
print("SHA-256:", hashlib.sha256(CLEAN.read_bytes()).hexdigest())

Cleaning log: 63 entries -> /Users/tiendung/Downloads/Intro_ds-main/docs/d2_outputs/d2_cleaning_log.csv
Clean dataset: 425,106 rows x 47 columns -> brfss2023_clean.parquet
SHA-256: 5415732cfd26e14f0a701faa644f9174a60e9715714c05370892356d104414ea


### Cleaning log (all decisions)

In [17]:
display(log_df.drop(columns="pct_of_rows").style.format({"rows_affected": "{:,}"}).hide(axis="index"))

step,variable,original,new_value,reason,rows_affected
C1,_STATE,"66 Guam, 72 Puerto Rico, 78 U.S. Virgin Islands",row removed,Outside the 48 states + DC in the file; results are described as U.S. adults,"8,217"
C2,all 29 columns,identical rows,none removed,Checked for duplicates; weight and PSU make true duplicates implausible,0
C3,"_AGE80, _LLCPWT, _BMI5",range check,no change,All values inside the plausible range; no row dropped,0
C4,_RFHYPE6 -> HTN_FLAG,1,0,No high blood pressure. CDC code 1 also holds 'pregnancy only' and 'borderline' answers (BPHIGH6 = 2 or 4); kept as 0 per CDC definition,"250,650"
C4,_RFHYPE6 -> HTN_FLAG,2,1,Told they have high blood pressure,"172,547"
C4,_RFHYPE6 -> HTN_FLAG,9,,"Unknown or missing: kept as NA, not coded as No","1,909"
C5,_RFCHOL3 -> CHOL_FLAG,1,0,Cholesterol not reported high,"215,131"
C5,_RFCHOL3 -> CHOL_FLAG,2,1,Told cholesterol is high,"155,586"
C5,_RFCHOL3 -> CHOL_FLAG,9,,"Unknown or missing: kept as NA, not coded as No","3,229"
C5,_RFCHOL3 -> CHOL_FLAG,blank,,"Never had cholesterol checked or unknown: NOT the same as normal cholesterol, kept as NA","51,160"


### Audit of the cleaning log
Every recoding step lists each code it touched with the number of rows, so the rows of one step must add up to the number of respondents in the data.

In [18]:
steps = ["C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13", "C15", "C16"]
aud = log_df[log_df["step"].isin(steps)].groupby("step", sort=False)["rows_affected"].sum().to_frame("rows_in_log")
aud["respondents_in_data"] = len(df)
aud["balanced"] = aud["rows_in_log"] == aud["respondents_in_data"]
display(aud.style.format({"rows_in_log": "{:,}", "respondents_in_data": "{:,}"}))
print(f"Every recoding step accounts for all {len(df):,} respondents:", bool(aud["balanced"].all()))

,rows_in_log,respondents_in_data,balanced
step,,,
C4,"425,106","425,106",True
C5,"425,106","425,106",True
C6,"425,106","425,106",True
C7,"425,106","425,106",True
C8,"425,106","425,106",True
C9,"425,106","425,106",True
C10,"425,106","425,106",True
C11,"425,106","425,106",True
C12,"425,106","425,106",True


Every recoding step accounts for all 425,106 respondents: True


### What D2 does and does not establish
* The tables above are **descriptive**. A higher reported-CVD share at higher `RiskCount` is a pattern in this survey, not evidence that the risk factors cause disease.
* **Self-reported:** every flag and every CVD answer is what the respondent said by telephone, not a clinical record.
* **Complete-case loss:** requiring all six flags drops respondents (row flow in section 5.1). Section 5.2 compares those dropped with those kept; any difference is a limitation to carry into the report.
* **Selection effect (section 5.2):** unknown cholesterol, BMI and smoking status remove mostly younger respondents, so the analysis sample is older and has a higher reported-CVD share than all adults. Compare groups; do not quote the sample level as a national rate.
* **Reported CVD means diagnosed already.** The outcome counts people who say they have been diagnosed; it does not measure undiagnosed disease.
* **Age** is strongly related to both risk factors and CVD, so the age-by-risk table (operation 5) is the first check; the formal confounding analysis is part of D3.
* Weights give population point estimates only; confidence intervals need `_STSTR` and `_PSU` (D3).
* **Scope:** results describe adults in the 48 states + DC present in the 2023 BRFSS public file (Kentucky and Pennsylvania were not released by CDC), not Vietnam and not every U.S. state.

**Next (D3):** statistical relationship with its interpretation, at least three charts, repository with README, environment file and seed.